In [ ]:
from importlib.resources import as_file, files
import json
import os
from pathlib import Path
import subprocess
import sys

run = Path(os.environ.get("OPENGHG_TUTORIAL_OUTPUT_PATH", "outputs/prior-check"))
resource = files("openghg_inversions.rhime").joinpath("config/standard_tutorial.ini")

def stage(command, *arguments):
    completed = subprocess.run(
        [sys.executable, "-m", "openghg_inversions.cli", command, *map(str, arguments)],
        capture_output=True, text=True,
    )
    if completed.returncode:
        raise RuntimeError(completed.stdout + completed.stderr)
    return completed.stdout

with as_file(resource) as config:
    stage("prepare", "--config", config, "--model", "standard",
          "--output-dir", run / "prepare")
prepared_path = run / "prepare/prepared-inputs.nc"
preparation_manifest = run / "prepare/prepare-manifest.json"
{
    "OpenGHG Inversions commit": os.environ.get("OPENGHG_TUTORIAL_CODE_REF", "local checkout"),
    "tutorial data": os.environ.get("OPENGHG_TUTORIAL_DATA_TAG", "v1.0.0"),
    "prepared inputs written": prepared_path.is_file(),
}

{'OpenGHG Inversions commit': 'ca468831c6603e4b368680953086c69a6b490d03',
 'tutorial data': 'v1.0.0',
 'prepared inputs written': True}

In [ ]:
with as_file(resource) as config:
    stage("prior-predictive", "--config", config, "--model", "standard",
          "--prepared-inputs", prepared_path,
          "--preparation-manifest", preparation_manifest,
          "--draws", 500, "--output-dir", run / "prior-predictive", "--strict")
readiness = json.loads((run / "prior-predictive/prior-predictive-readiness.json").read_text())
{"status": readiness["status"], "message": readiness["message"]}

{'status': 'pass', 'message': 'Prior predictive produced 500 finite draws.'}

In [ ]:
from openghg_inversions.serialization import load_trace
from openghg_inversions.inversion_data import RhimePreparedInputs

prior = load_trace(run / "prior-predictive/prior-predictive.nc")
prepared = RhimePreparedInputs.load(prepared_path)
observations = prepared.inv_inputs["mf"]
concentration_units = observations.attrs.get("units", "mole fraction")
{
    "groups": list(prior.children),
    "prior variables": sorted(prior["prior"].data_vars),
    "replicated observations": sorted(prior["prior_predictive"].data_vars),
}

{'groups': ['prior', 'prior_predictive', 'observed_data', 'constant_data'],
 'prior variables': ['bc', 'epsilon', 'mu', 'mu_bc', 'sigma', 'x'],
 'replicated observations': ['y']}

In [ ]:
import arviz_plots as azp
import matplotlib.pyplot as plt

azp.plot_ppc_dist(
    prior,
    group="prior_predictive",
    var_names=["y"],
    kind="ecdf",
    num_samples=100,
    backend="matplotlib",
    visuals={"remove_axis": False,
             "observed_dist": {"color": "black", "label": "Observed"}},
)
plt.gca().set_title("Prior predictive concentration distributions")
plt.gca().set_xlabel(f"Concentration ({concentration_units} mole fraction)")
plt.gca().set_ylabel("Cumulative probability")
plt.gca().legend()
plt.show()
"Prior predictive ECDFs: 100 simulated datasets and the observations."

'Prior predictive ECDFs: 100 simulated datasets and the observations.'

In [ ]:
import numpy as np

measurement_index = prepared.inv_inputs.indexes["nmeasure"]
sites = measurement_index.get_level_values("site")
times = measurement_index.get_level_values("time")
predictions = (
    prior["prior_predictive"]["y"]
    .stack(sample=("chain", "draw"))
    .transpose("sample", "nmeasure")
)

site_names = sites.unique().tolist()
figure, axes = plt.subplots(len(site_names), 1, figsize=(10, 7), squeeze=False)
for axis, site in zip(axes.flat, site_names):
    positions = np.flatnonzero(sites == site)
    intervals = predictions.isel(nmeasure=positions).quantile(
        [0.05, 0.5, 0.95], dim="sample"
    )
    axis.fill_between(
        times[positions],
        intervals.sel(quantile=0.05),
        intervals.sel(quantile=0.95),
        alpha=0.25,
        label="90% prior predictive interval",
    )
    axis.plot(times[positions], intervals.sel(quantile=0.5), label="median")
    trajectories = predictions.isel(sample=[0, 1, 2], nmeasure=positions)
    axis.plot(
        times[positions],
        trajectories.transpose("nmeasure", "sample"),
        color="0.5",
        alpha=0.5,
        linewidth=0.8,
    )
    axis.scatter(times[positions], observations.isel(nmeasure=positions), s=8,
                 label="observed")
    axis.set_title(site)
    axis.set_xlabel("Time")
    axis.set_ylabel(f"Concentration ({concentration_units})")
    axis.legend()
figure.autofmt_xdate()
figure.tight_layout()
plt.show()
{"sites": site_names, "concentration units": concentration_units}

{'sites': ['MHD', 'TAC'], 'concentration units': '1e-09'}

In [ ]:
with as_file(resource) as config:
    stage("sample", "--config", config, "--model", "standard",
          "--prepared-inputs", prepared_path,
          "--preparation-manifest", preparation_manifest,
          "--output-dir", run / "sample")
posterior = load_trace(run / "sample/posterior.nc")
{"posterior samples": {name: posterior["posterior"].sizes[name]
                       for name in ("chain", "draw")}}

{'posterior samples': {'chain': 2, 'draw': 50}}

In [ ]:
stage("diagnose", "--posterior", run / "sample/posterior.nc",
      "--sample-manifest", run / "sample/sample-manifest.json",
      "--output-dir", run / "diagnose")
diagnostics = json.loads((run / "diagnose/sampler-convergence.json").read_text())
{"status": diagnostics["status"], "message": diagnostics["message"]}

{'status': 'fail',
 'message': 'Posterior convergence thresholds were exceeded.'}

In [ ]:
azp.plot_ppc_dist(
    posterior,
    group="posterior_predictive",
    var_names=["y"],
    kind="ecdf",
    num_samples=100,
    backend="matplotlib",
    visuals={"remove_axis": False,
             "observed_dist": {"color": "black", "label": "Observed"}},
)
plt.gca().set_title("Posterior predictive distributions (smoke run only)")
plt.gca().set_xlabel(f"Concentration ({concentration_units} mole fraction)")
plt.gca().set_ylabel("Cumulative probability")
plt.gca().legend()
plt.show()
"Posterior predictive ECDFs: illustration only, not a converged scientific result."

'Posterior predictive ECDFs: illustration only, not a converged scientific result.'